In [4]:
%pip install openai -q

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import numpy as np
import os
from openai import OpenAI
from sklearn.metrics import cohen_kappa_score

from openai import OpenAI

client = OpenAI(api_key="API_KEY")
print("OpenAI client ready")

OpenAI client ready


In [9]:
import os
os.chdir('/Users/harshaggarwal/hinglish-sentiment-analysis')

train_custom = pd.read_csv("data/train_custom.csv")
dev_custom = pd.read_csv("data/dev_custom.csv")

full_df = pd.concat([train_custom, dev_custom]).reset_index(drop=True)
print("Total tweets:", len(full_df))
print(full_df['sentiment'].value_counts())
train_custom = pd.read_csv("data/train_custom.csv")
dev_custom = pd.read_csv("data/dev_custom.csv")

full_df = pd.concat([train_custom, dev_custom]).reset_index(drop=True)
print("Total tweets:", len(full_df))
print(full_df['sentiment'].value_counts())



Total tweets: 15480
sentiment
neutral     5787
positive    5139
negative    4554
Name: count, dtype: int64
Total tweets: 15480
sentiment
neutral     5787
positive    5139
negative    4554
Name: count, dtype: int64


In [10]:
sample_df = full_df.groupby('sentiment').apply(
    lambda x: x.sample(1000, random_state=42)
).reset_index(drop=True)

print("Sample size:", len(sample_df))
print(sample_df['sentiment'].value_counts())

Sample size: 3000
sentiment
negative    1000
neutral     1000
positive    1000
Name: count, dtype: int64


/var/folders/m8/j965yrg954b9_zblcxyfkbhm0000gn/T/ipykernel_86644/2869100790.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample_df = full_df.groupby('sentiment').apply(


In [11]:
EMOTIONS = ["joy", "sadness", "anger", "fear", "trust", "surprise"]

def get_emotion(tweet):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": """You are an emotion classifier for Hinglish (Hindi-English code-mixed) tweets.
Classify the tweet into exactly one of these 6 Plutchik emotions: joy, sadness, anger, fear, trust, surprise.
Return only the emotion label, nothing else."""
                },
                {
                    "role": "user",
                    "content": f"Tweet: {tweet}"
                }
            ],
            max_tokens=10,
            temperature=0
        )
        emotion = response.choices[0].message.content.strip().lower()
        if emotion not in EMOTIONS:
            return "unknown"
        return emotion
    except Exception as e:
        print(f"Error: {e}")
        return "unknown"

# Test on 1 tweet first
test_tweet = sample_df['text'].iloc[0]
print("Test tweet:", test_tweet)
print("Emotion:", get_emotion(test_tweet))

Test tweet: @ DrDeng9 @ theskindoctor13 @ ANI @ MamataOfficial Ye doctor homko chain se jeene nahi dega ore baba ami ki korbe sob borbod hogaya
Emotion: anger


In [12]:
from tqdm import tqdm

print("Labeling 3,000 tweets with GPT-4o-mini...")
emotions = []

for tweet in tqdm(sample_df['text']):
    emotion = get_emotion(tweet)
    emotions.append(emotion)

sample_df['emotion'] = emotions

print("\nDone!")
print("Emotion distribution:")
print(sample_df['emotion'].value_counts())
print("\nUnknown labels:", (sample_df['emotion'] == 'unknown').sum())

Labeling 3,000 tweets with GPT-4o-mini...


 13%|█▎        | 401/3000 [05:19<34:32,  1.25it/s] 


KeyboardInterrupt: 

In [13]:
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

def label_tweet(idx_tweet):
    idx, tweet = idx_tweet
    return idx, get_emotion(tweet)

print("Labeling 3,000 tweets in parallel...")
emotions = [''] * len(sample_df)

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(label_tweet, (i, tweet)): i 
               for i, tweet in enumerate(sample_df['text'])}
    
    for future in tqdm(as_completed(futures), total=len(sample_df)):
        idx, emotion = future.result()
        emotions[idx] = emotion

sample_df['emotion'] = emotions

print("\nDone!")
print("Emotion distribution:")
print(sample_df['emotion'].value_counts())
print("\nUnknown labels:", (sample_df['emotion'] == 'unknown').sum())

Labeling 3,000 tweets in parallel...


 25%|██▍       | 739/3000 [00:27<02:32, 14.82it/s]

Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type

 25%|██▍       | 746/3000 [00:28<01:47, 21.00it/s]

Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type

...

 49%|████▉     | 1470/3000 [01:10<01:13, 20.78it/s]

Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type

Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type

KeyboardInterrupt: 

Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type

In [14]:
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_emotion_with_retry(tweet, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system",
                        "content": """You are an emotion classifier for Hinglish (Hindi-English code-mixed) tweets.
Classify the tweet into exactly one of these 6 Plutchik emotions: joy, sadness, anger, fear, trust, surprise.
Return only the emotion label, nothing else."""
                    },
                    {
                        "role": "user",
                        "content": f"Tweet: {tweet}"
                    }
                ],
                max_tokens=10,
                temperature=0
            )
            emotion = response.choices[0].message.content.strip().lower()
            if emotion not in EMOTIONS:
                return "unknown"
            return emotion
        except Exception as e:
            if "429" in str(e):
                time.sleep(2 ** attempt)  # exponential backoff: 1s, 2s, 4s
            else:
                print(f"Error: {e}")
                return "unknown"
    return "unknown"

def label_tweet(idx_tweet):
    idx, tweet = idx_tweet
    return idx, get_emotion_with_retry(tweet)

print("Labeling 3,000 tweets...")
emotions = [''] * len(sample_df)

# Reduce workers to 10 to stay under rate limit
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(label_tweet, (i, tweet)): i 
               for i, tweet in enumerate(sample_df['text'])}
    
    for future in tqdm(as_completed(futures), total=len(sample_df)):
        idx, emotion = future.result()
        emotions[idx] = emotion

sample_df['emotion'] = emotions

print("\nDone!")
print("Emotion distribution:")
print(sample_df['emotion'].value_counts())
print("Unknown labels:", (sample_df['emotion'] == 'unknown').sum())

Labeling 3,000 tweets...


100%|██████████| 3000/3000 [22:48<00:00,  2.19it/s]


Done!
Emotion distribution:
emotion
anger       1350
joy         1092
sadness      244
trust        139
unknown      116
surprise      38
fear          21
Name: count, dtype: int64
Unknown labels: 116


In [15]:
# Save to CSV
sample_df.to_csv("data/emotion_labeled_sample.csv", index=False)
print("Saved!")

# Check unknowns
unknowns = sample_df[sample_df['emotion'] == 'unknown']
print("\nSample unknown tweets:")
print(unknowns['text'].head(5).tolist())

Saved!

Sample unknown tweets:
["@ ladywithflaws Mujhe samajh ni aya please clarify ma'am", 'The best form is you call me for leave ou to leave eis a quest ã o', 'And Jesus said unto him Go thy way ; thy faith hath made thee whole . And immediately he received his sight and fo … https // t . co / FGmIpZoLkY', '@ diljitdosanjh waaa pajii thoda ta mh bhut vada FAN haa i love ur voice sir ek rqst hai mehndi type song mh voice … https // t . co / xlE5WkZF61', '# SarfarazAhmed MASSIVE RESPECT to our Captain Sarfraz for wearing our national Dress World Cup jeeto ya haaro p … https // t co / LoDK6oTj9n']


In [16]:
sample_df_clean = sample_df[sample_df['emotion'] != 'unknown'].reset_index(drop=True)
sample_df_clean.to_csv("data/emotion_labeled_sample.csv", index=False)
print("Clean sample size:", len(sample_df_clean))
print(sample_df_clean['emotion'].value_counts())

Clean sample size: 2884
emotion
anger       1350
joy         1092
sadness      244
trust        139
surprise      38
fear          21
Name: count, dtype: int64


In [ ]:
# keep only 4 emotions with sufficient samples
valid_emotions = ['anger', 'joy', 'sadness', 'trust']
sample_df_4class = sample_df_clean[sample_df_clean['emotion'].isin(valid_emotions)].reset_index(drop=True)

print("4-class sample size:", len(sample_df_4class))
print(sample_df_4class['emotion'].value_counts())

sample_df_4class.to_csv("data/emotion_labeled_4class.csv", index=False)
print("Saved!")

4-class sample size: 2825
emotion
anger      1350
joy        1092
sadness     244
trust       139
Name: count, dtype: int64
Saved!


In [18]:
# Sample 200 tweets stratified across 4 emotions for manual verification
verify_df = sample_df_4class.groupby('emotion').apply(
    lambda x: x.sample(min(50, len(x)), random_state=42)
).reset_index(drop=True)

print("Verification sample:", len(verify_df))
print(verify_df['emotion'].value_counts())

# Save GPT labels separately — you'll add your labels manually
verify_df[['tweet_id', 'text', 'sentiment']].to_csv("data/manual_verification.csv", index=False)
print("Saved to manual_verification.csv — add your labels in a new column called 'my_emotion'")


Verification sample: 200
emotion
anger      50
joy        50
sadness    50
trust      50
Name: count, dtype: int64
Saved to manual_verification.csv — add your labels in a new column called 'my_emotion'


/var/folders/m8/j965yrg954b9_zblcxyfkbhm0000gn/T/ipykernel_86644/3707147569.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  verify_df = sample_df_4class.groupby('emotion').apply(


In [19]:
# Add GPT labels to verification df for kappa calculation
verify_df[['tweet_id', 'text', 'sentiment', 'emotion']].to_csv(
    "data/gpt_verification_labels.csv", index=False
)
print("GPT labels saved separately")

GPT labels saved separately


In [21]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# Load both files
manual_df = pd.read_csv("data/manual_verification.csv")
gpt_df = pd.read_csv("data/gpt_verification_labels.csv")

# Merge on tweet_id
merged = manual_df.merge(gpt_df[['tweet_id', 'emotion']], on='tweet_id')

print("Merged size:", len(merged))
print("\nYour label distribution:")
print(merged['my_emotion'].value_counts())
print("\nGPT label distribution:")
print(merged['emotion'].value_counts())

# Filter out 'other' labels
merged_clean = merged[
    (merged['my_emotion'].isin(['anger', 'joy', 'sadness', 'trust'])) &
    (merged['emotion'].isin(['anger', 'joy', 'sadness', 'trust']))
]
print("\nClean merged size:", len(merged_clean))

# Compute kappa
kappa = cohen_kappa_score(merged_clean['my_emotion'], merged_clean['emotion'])
print("\nCohen's Kappa:", round(kappa, 4))

Merged size: 200

Your label distribution:
my_emotion
trust      71
anger      58
sadness    36
joy        35
Name: count, dtype: int64

GPT label distribution:
emotion
anger      50
joy        50
sadness    50
trust      50
Name: count, dtype: int64

Clean merged size: 200

Cohen's Kappa: 0.6667


In [ ]:
# Save final emotion labeled dataset
sample_df_4class.to_csv("data/emotion_labeled_final.csv", index=False)
print("Final dataset saved:", len(sample_df_4class), "tweets")
print(sample_df_4class['emotion'].value_counts())

Final dataset saved: 2825 tweets
emotion
anger      1350
joy        1092
sadness     244
trust       139
Name: count, dtype: int64


In [23]:
import pandas as pd
df = pd.read_csv("data/emotion_labeled_final.csv")
print(df.columns.tolist())
print(df.head(3))

['tweet_id', 'sentiment', 'text', 'emotion']
   tweet_id sentiment                                               text  \
0     28221  negative  @ DrDeng9 @ theskindoctor13 @ ANI @ MamataOffi...   
1       414  negative  @ waseemAli0007 @_ Mansoor _ Ali @ ImranKhanPT...   
2     29875  negative  @ UrmileshJ Kuchh nhi hoga media wale aur nich...   

  emotion  
0   anger  
1   anger  
2   anger  
